In [1]:
import fitz  # PyMuPDF

class PDFSegmentation:
    def __init__(self, pdf_path, area_threshold=0.3):
        """
        :param area_threshold: Percentage of page area (0.0 to 1.0) 
                               that must be covered to consider it a flowchart.
        """
        self.doc = fitz.open(pdf_path)
        self.area_threshold = area_threshold

    def is_flowchart_page(self, page):
        """
        Heuristic: Calculate the ratio of drawing/image area to total page area.
        """
        page_rect = page.rect
        total_area = page_rect.width * page_rect.height
        
        # Get all drawing objects (paths) and images
        drawings = page.get_drawings()
        images = page.get_images()
        
        covered_area = 0
        
        # Calculate area of drawing paths (boxes, lines, etc.)
        for path in drawings:
            # We look at the bounding box of the drawing path
            rect = path['rect']
            covered_area += (rect.width * rect.height)
            
        # Calculate area of images (like diagrams embedded as images)
        for img in images:
            # get_image_rects gives the position on the page
            img_rects = page.get_image_rects(img[0])
            for rect in img_rects:
                covered_area += (rect.width * rect.height)
        
        ratio = covered_area / total_area
        return ratio > self.area_threshold, ratio

    def get_flowchart_pages(self):
        flowchart_pages = []
        for page_num in range(len(self.doc)):
            page = self.doc[page_num]
            is_match, ratio = self.is_flowchart_page(page)
            
            if is_match:
                flowchart_pages.append(page_num + 1)
                print(f"Page {page_num + 1} identified as potential flowchart (Coverage: {ratio:.2%})")
        
        return flowchart_pages

# --- Execution ---
# if __name__ == "__main__":
#     segmenter = PDFSegmentation("your_document.pdf", area_threshold=0.25)
#     pages = segmenter.get_flowchart_pages()
#     print(f"\nFlowcharts found on pages: {pages}")